In [1]:
!pip3 install mysql-connector-python python-dotenv

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv
import os

In [9]:
# Load environment variables from .env file
load_dotenv()

# Database connection details
db_host = os.getenv('DB_HOST', 'localhost')
db_port = os.getenv('DB_PORT', '3307')
db_user = os.getenv('MYSQL_USER', 'root')
db_password = os.getenv('MYSQL_PASSWORD', 'password')
db_database = os.getenv('MYSQL_DATABASE', 'db_sqak')

In [ ]:
def create_connection(host_name, user_name, user_password, db_name, port_name=3306):
    connection = None
    try:
        connection = mysql.connector.connect(
            host=host_name,
            user=user_name,
            port=port_name,
            passwd=user_password,
            database=db_name
        )
        print("Connection to MySQL DB successful")
    except Error as e:
        print(f"The error '{e}' occurred")
    return connection

In [11]:
def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print(f"Query executed successfully: {query}")
    except Error as e:
        print(f"The error '{e}' occurred when executing: {query}")

In [12]:
def execute_script_from_file(connection, file_path):
    cursor = connection.cursor()
    with open(file_path, 'r') as file:
        script = file.read()

    statements = script.split(';')

    for statement in statements:
        if statement.strip():
            try:
                cursor.execute(statement)
                connection.commit()
                print(f"Statement executed successfully: {statement}")
            except Error as e:
                if e.errno == 1050:  # Error code for "Table already exists"
                    print(f"The error '{e}' occurred when executing statement: {statement} - Table already exists, continuing...")
                elif e.errno == 1061:  # Error code for "Duplicate key name"
                    print(f"The error '{e}' occurred when executing statement: {statement} - Duplicate key name, continuing...")
                elif e.errno == 1062:  # Error code for "Duplicate entry"
                    print(f"The error '{e}' occurred when executing statement: {statement} - Duplicate entry, continuing...")
                elif e.errno == 1146:  # Error code for "Table doesn't exist"
                    print(f"The error '{e}' occurred when executing statement: {statement} - Table doesn't exist, continuing...")
                else:
                    print(f"The error '{e}' occurred when executing statement: {statement}")

In [ ]:
def main():
    connection = create_connection(db_host, db_user, db_password, db_database, db_port)

    if connection:
        execute_script_from_file(connection, "1create.sql")
        execute_script_from_file(connection, "3insert.sql")

        connection.close()

if __name__ == "__main__":
    main()

The error '2003: Can't connect to MySQL server on 'localhost:3306' (10061 No connection could be made because the target machine actively refused it)' occurred
